# Biodiversity Intactness Index (BII)

This notebook loads Biodiversity Intactness Index (BII) raster data and resamples it to a chosen master grid. It than creates:
- a raster at a chosen pixel size
- a map figure (`OUT_PNG`)

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `bii.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)


In [ ]:
#import packages 
import pandas as pd
import geopandas as gpd
import fiona
import pycountry
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
import math
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import rasterio.windows
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.transform import Affine



In [ ]:
# Configuration (edit these paths if needed)

in_path = "bii.tif"

# Vector country boundaries (must contain an ISO/area code column matching `area_code` from the CSV)
COUNTRIES_GPKG = "World_Countries_(Generalized)_8414823838130214587.gpkg"


# MASTER GRID 
MASTER_CRS = "EPSG:6933"
MASTER_WIDTH = 6948
MASTER_HEIGHT = 2928
MASTER_TRANSFORM = Affine(5000.0, 0.0, -17367529.34,
                          0.0, -5000.0,  7296404.55)
DST_NODATA = -9999.0

out_path = "bii_5000m.tif"
OUT_PNG = "bii.png"


In [ ]:
resampling = Resampling.bilinear  # continuous

profile = {
    "driver": "GTiff",
    "crs": MASTER_CRS,
    "transform": MASTER_TRANSFORM,
    "width": MASTER_WIDTH,
    "height": MASTER_HEIGHT,
    "count": 1,
    "dtype": "float32",
    "nodata": DST_NODATA,
    "tiled": True,
    "blockxsize": 1024,
    "blockysize": 1024,
    "compress": "lzw",
}

with rasterio.open(in_path) as src, rasterio.open(out_path, "w", **profile) as dst:
    for _, window in dst.block_windows(1):
        out_block = np.full((window.height, window.width), DST_NODATA, dtype=np.float32)
        win_transform = rasterio.windows.transform(window, MASTER_TRANSFORM)

        reproject(
            source=rasterio.band(src, 1),
            destination=out_block,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=win_transform,
            dst_crs=MASTER_CRS,
            dst_nodata=DST_NODATA,
            resampling=resampling,
        )

        # scale + invert, keep nodata
        m = np.isfinite(out_block) & (out_block != DST_NODATA)
        out_block[m] = 1.0 - (out_block[m] / 100.0)

        dst.write(out_block, 1, window=window)

print(out_path)

In [ ]:
#plot

#paths
raster_path = out_path
countries_path = COUNTRIES_GPKG
out_png = OUT_PNG

max_width = 5000

#open raster
with rasterio.open(raster_path) as src:
    bounds = src.bounds
    crs = src.crs
    nodata_val = src.nodata

    #calculate display size
    scale = max_width / src.width
    out_w = int(src.width * scale)
    out_h = int(src.height * scale)
    
    #read raster band with new size using .nearest resampling
    arr = src.read(
        1,
        out_shape=(out_h, out_w),
        resampling=Resampling.nearest
    ).astype("float32")
    #adjust transform to new size
    transform = src.transform * src.transform.scale(
        src.width / out_w,
        src.height / out_h
    )
#open country boundaries gpkg and drop Antarctica 
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"]

#build country mask
country_mask = geometry_mask(
    geometries=world.geometry,
    transform=transform,
    invert=True,
    out_shape=arr.shape
)

#define nodata
nodata = (arr == -9999)

#convert -9999 to nan for plotting
arr = arr.astype("float32")
arr[arr == -9999] = np.nan

#mask data outside country and no data for arr
masked = np.ma.masked_where((~country_mask) | nodata, arr)
#transparency settings
alpha = np.where(country_mask, 0.95, 0.0)

#define valid values
valid = arr[country_mask & np.isfinite(arr)]

# robust min/max from valid pixels
vmin = float(valid.min())
vmax = float(valid.max())

nodata_color = "#D1D5DB"
cmap = LinearSegmentedColormap.from_list(
    "bii_red_grad",
    ["#FFF5F0", "#FCBBA1", "#EF3B2C", "#99000D"]
)
cmap.set_bad(color=nodata_color)

#build figure, set size and background color
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

#plot country boundaries, set color and linewidth
world.plot(
    ax=ax,
    facecolor="#F6F7F9",
    edgecolor="#B9C0C8",
    linewidth=0.35,
    zorder=1
)

#plot raster
raster = ax.imshow(
    masked,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    interpolation="nearest",
    alpha=alpha, #transparency
    zorder=2 #plot on top of country boundaries
)

# Plot country boundaries on top
world.boundary.plot(ax=ax, color="#695C5A", linewidth=0.3, zorder=2)


#set title
ax.set_title(
    "Biodiversity intactness",
    fontsize=18,
    fontweight="semibold",
    pad=14
)
#no axis
ax.set_axis_off()

# NoData legend patch 
legend_handles = [Patch(facecolor=nodata_color, edgecolor="none", label="No data")]
leg = ax.legend(
    handles=legend_handles,
    loc="lower left",
    frameon=True,
    framealpha=1,
    facecolor="white",
    edgecolor="#E3E6EA",
    borderpad=0.8,
    handlelength=1.2,
)
#create colorbar
cbar = plt.colorbar(raster, ax=ax, fraction=0.03, pad=0.02) #width of color bar relative to plot and space between plot and colorbar
cbar.set_label(
    "Biodiversity intactness (inverted)",
    fontsize=16,
    color="#2B2F36"
)
cbar.ax.tick_params(labelsize=10, colors="#2B2F36") #numbers on colorbar
cbar.outline.set_edgecolor("#E3E6EA") #edgecolor of colorbar
cbar.outline.set_linewidth(1.0)
cbar.ax.set_facecolor("white")

plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
